# RU OCR: speed benchmark — Paddle (GPU, CPU), ONNX Runtime, OpenVINO

The `CKPT` checkpoint of `CKPT_DATASET` is exported (Paddle inference model, and ONNX from a CPU export) and read
on the lines of `LABELS` by each backend through PaddleX `create_model`, as a user would run it:

| Backend | Model | Device |
|---|---|---|
| `paddle gpu` | Paddle inference | one GPU (only in a GPU session) |
| `paddle cpu` | Paddle inference | CPU |
| `onnxruntime cpu` | ONNX | CPU, `CPUExecutionProvider` |
| `openvino cpu` | ONNX | CPU, ONNX Runtime `OpenVINOExecutionProvider` |

- With `SORT_BY_WIDTH` every batch size above 1 runs twice: lines in file order, and sorted by width/height so
  that a batch holds lines of similar width (each batch is padded to its widest line).
- The measuring is `ru_ocr/scripts/benchmark.py` (the same runs outside Kaggle on any exported model); it prints
  the hardware first and one line per run. Each backend × batch size runs in its own process. The images are decoded into memory first, so disk and
  decoding are not timed; the first `WARMUP` calls are not timed either.
- Measured: lines/s, ms per line, ms per `predict` call (one batch: mean, p50, p95), model load time, and accuracy on
  the same lines (exact match, spaces count) as a check that every backend reads the same.
- Thread counts are the libraries' defaults; the hardware header says how many cores the process may use.

**Output** (`/kaggle/working`): `benchmark.md` (hardware header + table), `benchmark.tsv`, `hardware.json`.

**Input**: the checkpoint dataset (`ru-ocr-checkpoints-ft4`) and `ru-ocr-val5k-tar` (the 5 000 crops of `all_val5k`
in one `.tarpack`, 26 MB, with `all_val5k_w_h.txt` and `all_val5k_clean_w_h.txt`). Smoke run: attach `ru-ocr-smoke` (100 lines of `all_val5k_clean_w_h.txt` with their crops) instead of the full
crops and labels. **Settings**: Accelerator GPU T4 x2 (all four backends) or None (CPU backends only), Internet on.
No secret needed. Save & Run All.

In [ ]:
REPO = "https://github.com/ErshovVE/PaddleOCR.git"  # cloned for the config and tools/export_model.py
BRANCH = "feat/ru-ppocrv6-small-training"
CKPT = "best_accuracy"   # checkpoint prefix in CKPT_DATASET
CKPT_DATASET = "ru-ocr-checkpoints-ft4"
CONFIG = "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml"
MODEL_NAME = "PP-OCRv6_small_rec"
LABELS = "all_val5k_clean_w_h.txt"  # found anywhere under /kaggle/input (ru-ocr-labels or ru-ocr-smoke)
LIMIT = None             # only the first LIMIT lines (None = all)
BATCHES = [1, 4, 8]      # batch sizes, each a separate run
SORT_BY_WIDTH = True     # batches above 1 also run on lines sorted by width/height (similar widths per batch)
WARMUP = 3               # predict calls before timing starts
BACKENDS = ["paddle gpu", "paddle cpu", "onnxruntime cpu", "openvino cpu"]  # "paddle gpu" is skipped without a GPU
MODEL_DIR = "/kaggle/working/model"
import os, re, shutil, subprocess, sys, time
HAS_GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
print("GPU session" if HAS_GPU else "CPU session: paddle gpu is skipped")

## Paddle, PaddleX, ONNX Runtime with OpenVINO, code

In [ ]:
import shlex, signal, threading


def sh(cmd, timeout=1200, attempts=3):
    """Run a shell command with its output in the cell. Some installs stall forever with an idle CPU: a run is
    killed with its whole process group after `timeout` seconds and retried."""
    for attempt in range(1, attempts + 1):
        print(time.strftime("%H:%M:%S"), f"[{attempt}/{attempts}]", cmd, flush=True)
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                start_new_session=True)
        timer = threading.Timer(timeout, os.killpg, (proc.pid, signal.SIGKILL))
        timer.start()
        for line in proc.stdout:
            print(line, end="", flush=True)
        hung = not timer.is_alive()
        timer.cancel()
        if proc.wait() == 0:
            return
        print(time.strftime("%H:%M:%S"), f"hung for {timeout} s, killed" if hung else f"exit code {proc.returncode}", flush=True)
    raise RuntimeError(f"failed {attempts} times: {cmd}")


if HAS_GPU:
    paddle_pkg = "paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/"
else:
    paddle_pkg = "paddlepaddle==3.3.1"
PIP = f"{shlex.quote(sys.executable)} -m pip install --progress-bar off --timeout 120 --retries 10"
from importlib.metadata import PackageNotFoundError, version
try:
    IMAGE_NCCL = version("nvidia-nccl-cu12")
except PackageNotFoundError:
    IMAGE_NCCL = None
sh(f"{PIP} {paddle_pkg}", attempts=5)
# paddlepaddle-gpu (cu126) pins an older nvidia-nccl-cu12 over the image's one, and the image's torch then fails on
# import (paddlex imports torch through modelscope): the image's version goes back
if HAS_GPU and IMAGE_NCCL and version("nvidia-nccl-cu12") != IMAGE_NCCL:
    sh(f"{PIP} --no-deps --force-reinstall nvidia-nccl-cu12=={IMAGE_NCCL}")
sh(f"{PIP} 'paddlex[ocr-core]>=3.7,<3.8' onnxruntime-openvino")
# ONNX: paddle2onnx has wheels only for Python <= 3.12 and does not take an export made with paddlepaddle-gpu, so
# a CPU paddle in a Python 3.12 venv exports the checkpoint once more for the conversion
sh(f"{PIP} uv")
sh("uv venv --clear --seed --python 3.12 /tmp/py312")
sh("uv pip install --python /tmp/py312/bin/python paddlepaddle==3.3.1 'paddlex[ocr-core]>=3.7,<3.8'")
sh("/tmp/py312/bin/paddlex --install paddle2onnx")
import onnxruntime
print("ONNX Runtime providers:", onnxruntime.get_available_providers())
res = subprocess.run([sys.executable, "-c", "import torch, paddle; print('torch', torch.__version__, 'paddle', "
                      "paddle.__version__, 'cuda', paddle.device.is_compiled_with_cuda())"], capture_output=True, text=True)
print(res.stdout.strip(), res.stderr.strip()[-2000:] if res.returncode else "")
assert res.returncode == 0, "torch or paddle does not import, see above"

In [ ]:
%cd /kaggle/working
!rm -rf PaddleOCR && git clone -q --depth 1 --branch {BRANCH} {REPO}
%cd /kaggle/working/PaddleOCR
!git log --oneline -1
sh(f"{PIP} -r requirements.txt")
sh("uv pip install --python /tmp/py312/bin/python -r requirements.txt")  # tools/export_model.py in the venv

## Data, labels, weights
Crop archives `*.tarpack` are unpacked into `/tmp/ru_root`; folders of single crops (`<...>/stroyinf/<folder>` with
`crops/`) are linked there.

In [ ]:
ROOT = "/tmp/ru_root"
os.makedirs(os.path.join(ROOT, "stroyinf"), exist_ok=True)
for top, dirs, files in os.walk("/kaggle/input"):
    for name in sorted(f for f in files if f.endswith(".tarpack")):
        start = time.time()
        subprocess.run(["tar", "-xf", os.path.join(top, name), "-C", ROOT], check=True)
        print(f"unpacked {name} in {time.time() - start:.0f} s", flush=True)
labels_path, weights = None, []
for top, dirs, files in os.walk("/kaggle/input"):
    if "crops" in dirs:
        dirs.remove("crops")
        if os.path.basename(os.path.dirname(top)) == "stroyinf":
            link = os.path.join(ROOT, "stroyinf", os.path.basename(top))
            if not os.path.exists(link):
                os.symlink(top, link)
    if LABELS in files:
        labels_path = os.path.join(top, LABELS)
    if CKPT + ".pdparams" in files and CKPT_DATASET in top.split(os.sep):
        weights.append(os.path.join(top, CKPT))
assert labels_path and len(weights) == 1, f"attach a dataset with {LABELS} and {CKPT_DATASET}: {labels_path}, {weights}"
WEIGHTS = weights[0]
with open(labels_path, encoding="utf-8") as f:
    n_lines = sum(1 for line in f if line.strip())
print("labels:", labels_path, n_lines, "lines", f"(first {LIMIT} used)" if LIMIT else "")
print("weights:", WEIGHTS)

## Export: checkpoint -> Paddle inference model, ONNX
`model/inference` is exported by the session's paddle (GPU build in a GPU session) and serves both Paddle backends;
`model/inference_cpu` is exported by the CPU paddle of the venv only to be converted into `model/onnx`.

In [ ]:
export = ["tools/export_model.py", "-c", CONFIG, "-o", f"Global.pretrained_model={WEIGHTS}"]
subprocess.run([sys.executable, *export, f"Global.save_inference_dir={MODEL_DIR}/inference"], check=True,
               stdout=subprocess.DEVNULL)
subprocess.run(["/tmp/py312/bin/python", *export, "Global.use_gpu=false",
                f"Global.save_inference_dir={MODEL_DIR}/inference_cpu"], check=True, stdout=subprocess.DEVNULL)
# PATH: paddlex looks for the paddle2onnx executable of the venv
env = dict(os.environ, PATH="/tmp/py312/bin:" + os.environ["PATH"])
subprocess.run(["paddlex", "--paddle2onnx", "--paddle_model_dir", f"{MODEL_DIR}/inference_cpu",
                "--onnx_model_dir", f"{MODEL_DIR}/onnx"], env=env, check=True)
for d in ("inference", "onnx"):
    print(d, sorted(os.listdir(f"{MODEL_DIR}/{d}")))

## Benchmark
`ru_ocr/scripts/benchmark.py`: hardware header, then each backend × batch size in its own process; results in
`/kaggle/working` (`benchmark.md`, `benchmark.tsv`, `hardware.json`, `run_*.json` with the texts of each run).

In [ ]:
args = [sys.executable, "ru_ocr/scripts/benchmark.py", "--labels", labels_path, "--root", ROOT,
        "--paddle-model", f"{MODEL_DIR}/inference", "--onnx-model", f"{MODEL_DIR}/onnx", "--model-name", MODEL_NAME,
        "--backends", *BACKENDS, "--batches", *map(str, BATCHES), "--warmup", str(WARMUP), "--limit", str(LIMIT or 0),
        "--out", "/kaggle/working", "--note", f"Model: {CKPT_DATASET}/{CKPT} ({MODEL_NAME}); ONNX from a CPU export"]
if not SORT_BY_WIDTH:
    args.append("--no-sort")
sh(shlex.join(args), timeout=6 * 3600, attempts=1)